In [1]:
import sys
sys.version_info

sys.version_info(major=3, minor=6, micro=2, releaselevel='final', serial=0)

In [ ]:
def relu(x):
    if x > 0: return x
    else: return 0

In [6]:
from itertools import permutations, combinations, product

l = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j']

In [8]:
res = []
for combination in [set(c) for c in permutations(l, 3)]:
    if combination not in res: res.append(combination)
res

[{'a', 'b', 'c'},
 {'a', 'b', 'd'},
 {'a', 'b', 'e'},
 {'a', 'b', 'f'},
 {'a', 'b', 'g'},
 {'a', 'b', 'h'},
 {'a', 'b', 'i'},
 {'a', 'b', 'j'},
 {'a', 'c', 'd'},
 {'a', 'c', 'e'},
 {'a', 'c', 'f'},
 {'a', 'c', 'g'},
 {'a', 'c', 'h'},
 {'a', 'c', 'i'},
 {'a', 'c', 'j'},
 {'a', 'd', 'e'},
 {'a', 'd', 'f'},
 {'a', 'd', 'g'},
 {'a', 'd', 'h'},
 {'a', 'd', 'i'},
 {'a', 'd', 'j'},
 {'a', 'e', 'f'},
 {'a', 'e', 'g'},
 {'a', 'e', 'h'},
 {'a', 'e', 'i'},
 {'a', 'e', 'j'},
 {'a', 'f', 'g'},
 {'a', 'f', 'h'},
 {'a', 'f', 'i'},
 {'a', 'f', 'j'},
 {'a', 'g', 'h'},
 {'a', 'g', 'i'},
 {'a', 'g', 'j'},
 {'a', 'h', 'i'},
 {'a', 'h', 'j'},
 {'a', 'i', 'j'},
 {'b', 'c', 'd'},
 {'b', 'c', 'e'},
 {'b', 'c', 'f'},
 {'b', 'c', 'g'},
 {'b', 'c', 'h'},
 {'b', 'c', 'i'},
 {'b', 'c', 'j'},
 {'b', 'd', 'e'},
 {'b', 'd', 'f'},
 {'b', 'd', 'g'},
 {'b', 'd', 'h'},
 {'b', 'd', 'i'},
 {'b', 'd', 'j'},
 {'b', 'e', 'f'},
 {'b', 'e', 'g'},
 {'b', 'e', 'h'},
 {'b', 'e', 'i'},
 {'b', 'e', 'j'},
 {'b', 'f', 'g'},
 {'b', 'f'

In [7]:
def get_k(n):
    k = 0
    while 2 ** k < n + k + 1:
        k += 1
    return k

def get_check_groups(n, k):
    check_groups = {}
    for i in range(1, k + 1):
        index = 2 ** (i - 1)
        check_groups[index] = []
        for j in range(index + 1, n + k + 1):
            if j & (1 << (i - 1)):
                check_groups[index].append(j)
    return check_groups

def get_check_code(group_data, parity):
    if parity == 'odd':
        return 1 - sum(group_data) % 2
    elif parity == 'even':
        return sum(group_data) % 2
    else:
        raise ValueError('parity must be odd or even')

class HammingCode:
    def __init__(self, data: (list, tuple, str, int)=None):
        if data is not None:
            self._process(data)

    def _process(self, data: (list, tuple, str, int)):
        if isinstance(data, str):
            data = [int(i) for i in data]
        elif isinstance(data, int):
            # 转为二进制列表
            data = [int(i) for i in bin(data)[2:]]
        self.data = data
        self.n = len(data)
        self.k = get_k(self.n)
        self.check_groups = get_check_groups(self.n, self.k)
    
    def _inverse_print(self, code):
        print("Hamming Code: {}".format("".join([str(i) for i in code[::-1]])))

    def _check_error(self, code=None):
        if code is None:
            code = self.code
        errors = []
        for i, group in self.check_groups.items():
            res = get_check_code([code[j - 1] for j in group + [i]], 'odd')
            if res: # 有错
                errors.append(1) # 补1
            else:
                errors.append(0) # 补0
        if sum(errors) == 0:
            return None
        else:
            return sum([2 ** i for i, e in enumerate(errors) if e])

    def _check_error2(self, code=None):
        if code is None:
            code = self.code
        errors = set([i for i in range(1, self.n + self.k + 1)])
        for i, group in self.check_groups.items():
            res = get_check_code([code[j - 1] for j in group + [i]], 'odd')
            if res: # 有错
                pass
            else:
                errors -= set(group + [i])

        if len(errors) == 0:
            return None
        else:
            return errors

    def __call__(self, data=None):
        if data is not None:
            self._process(data)
        return self.encode()
    
    def encode(self):
        # 默认奇校验
        code = [0] * (self.n + self.k)
        data_index = 0
        for i in range(1, self.n + self.k + 1):
            if i in self.check_groups:
                code[i - 1] = None
            else:
                code[i - 1] = self.data[self.n - data_index - 1]
                data_index += 1
        for i, group in self.check_groups.items():
            code[i - 1] = get_check_code([code[j - 1] for j in group], 'odd')
        # 倒序输出字符串
        self._inverse_print(code)
        self.code = code
        return code
    
    def decode(self, code=None):
        if code is None:
            code = self.code
        # 默认奇校验
        errors = self._check_error(code)
        if errors is not None:
            code[errors - 1] = 1 - code[errors - 1]
        data = []
        for i in range(1, self.n + self.k + 1):
            if i not in self.check_groups:
                data.append(code[i - 1])
        self.data = data[::-1]
        return "".join([str(i) for i in data[::-1]])

In [8]:
ham = HammingCode() # 从高位到低位，称之为大端序

In [33]:
hc = ham("111001")
hc

Hamming Code: 1111000100


[0, 0, 1, 0, 0, 0, 1, 1, 1, 1]

In [34]:
hc[-1] = 0
hc

[0, 0, 1, 0, 0, 0, 1, 1, 1, 0]

In [35]:
ham._check_error(hc)

10

In [36]:
ham._check_error2(hc)

{2, 8, 10}